In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class DynamicGraph(nn.Module):
    def __init__(self, feature_dim=64, max_nodes=1024, max_edges_per_node=8, device=None):
        super().__init__()
        if device==None:
            if hasattr(torch, "xpu") and torch.xpu.is_available():
                device = torch.device("xpu")
            elif torch.cuda.is_available():
                device = torch.device("cuda")
            else:
                device = torch.device("cpu")
        self.device = torch.device(device)
        print(f"\nUsing device : {device}\n")

        self.D = feature_dim
        self.max_nodes = max_nodes
        self.max_edges = max_nodes * max_edges_per_node
        # max_edges = self.max_edges
        self.max_edges_per_node = max_edges_per_node

        ## Node Thingys

        # Node Weights (no_of_nodes,1)
        self.node_weight = nn.Parameter(
            torch.zeros(
                max_nodes,
                feature_dim,
                device=self.device
            )
        )

        # Node Bias (no_of_nodes,1)
        self.node_bias = nn.Parameter(
            torch.randn(
                max_nodes,
                feature_dim,
                device=self.device
            ) * 0.02
        )

        # Node Active/Deactive (no_of_nodes,1)
        self.register_buffer(
            "node_alive",
            torch.zeros(
                max_nodes,
                dtype=torch.bool,
                device=self.device
            )
        )

        # Node Activation Type (Relu/Sigmoid/LRelu) (no_of_nodes,1)
        self.register_buffer(
            "node_activation",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )   
        )

        # Node Out Degree (no_of_nodes,1) (Value ranges from 0 to max_edges_per_node)
        self.register_buffer(
            "out_degree",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )
        )

        # Node In Degree (no_of_nodes,1) (Value ranges from 0 to no_of_nodes * max_edges_per_node)
        self.register_buffer(
            "in_degree",
            torch.zeros(
                max_nodes,
                dtype=torch.long,
                device=self.device
            )
        )

        # No of current nodes (Includes Active + Deactive)
        self.nodes_count = 0


        ## Edge Thingys

        # Edge Source[edge_source][edge_destination] (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_src",
            torch.zeros(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Index[edge_destination][edge_source] (no_of_nodes,max_edges_per_node)
        # Used to store the index of edge stored in edge_src matrix for quick operation in back prop
        self.register_buffer(
            "edge_index",
            torch.zeros(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Destination[edge_destination][edge_source] (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_dest",
            torch.zeros(
                max_nodes,
                max_edges_per_node,
                dtype=torch.long,
                device=self.device
            )
        )

        # Edge Weight[edge_source][edge_destination] (no_of_nodes,max_edges_per_node)
        self.edge_weight = nn.Parameter(
            torch.randn(
                max_nodes,
                max_edges_per_node,
                device=self.device
            ) * 0.02
        )

        # Edge Active/Deactive[edge_source][edge_destination] (no_of_nodes,max_edges_per_node)
        self.register_buffer(
            "edge_alive",
            torch.zeros(
                max_nodes,
                max_edges_per_node,
                dtype=torch.bool,
                device=self.device
            )   
        )

        @torch.no_grad()
        def add_node(self, activation=1):
            if self.nodes_count >= self.max_nodes:
                raise RuntimeError("Node capacity reached in Add Node"+str(self.max_nodes))
            index = self.nodes_count
            self.node_alive[index] = True
            self.node_activation[index] = activation
            self.nodes_count += 1
            return index

        @torch.no_grad()
        def remove_node(self, index):
            if index>=self.nodes_count or index<0:
                raise RuntimeError("Out of bound in Remove Node")

            self.node_alive[index] = False




In [10]:
temp = DynamicGraph()



Using device : xpu

